# Benchmark repeats: seeds32 and33

Enable Internet and one GPU on Kaggle. Run stages in order; no attached
input bundle is required for these four automatically downloaded datasets.
Source loads latest main from chahineNejm/Generalized-Patchwork in fresh
workers. No model/data implementations are embedded here.

This finite batch has44 runs: California Housing12, PROTEIN12, TCO12,
OISST8. Seeds32 and33 vary model initialization/training only. Data splits
and preprocessing stay seed31. Existing seed31 results are not rerun,
automatically loaded or pooled. All settings and seed identities are saved.

Retain the tested configurations: SVGP, GPW8/4 patches and ProSpar8/4
experts for California/PROTEIN/TCO, plus reduced-data PC-tree Patchwork.
OISST retains its tested SVGP, GPW8, ProSpar8 and PC-tree Patchwork only.
Total budgets:512 for California/OISST,1024 for PROTEIN/TCO. This is not
the1024-point stitching/rank ablation. No VNNGP, FLIGHT or other datasets.

Standard Patchwork uses the primary eight-patch GPW initial observed rows
for each model seed and their targets; it does not train on all targets.
GPW/SVGP/ProSpar train on all training observations. Keep the existing
ARD RBF kernels, initialization, bounds, numerics, learning rates,
ProSpar T=1/c=1/nuggets=0.01/prior_jitter=0 and other tested settings.
Those explicit ProSpar overrides match the saved seed31 archive; they
do not change the newer general defaults T=0.1/c=2. Do not pool results
from different ProSpar starts as identical configurations.

EPOCHS=100 maximum, WARMUP_EPOCHS=20; training-loss early stopping has
patience10 and relative minimum improvement0.001 after warm-up. Evaluate
the last state, not a restored best checkpoint. For a smoke test explicitly
set EPOCHS=3 and WARMUP_EPOCHS=1. Review the plan, then RUN_TRAINING=True.

Dataset/model/seed labels refresh every20 epochs plus initial/final state.
Full loss history stays per epoch; visuals update one existing display.
Completed metrics and ZIPs save before each next run. Results live under
/kaggle/working/gpw_results. Download archives or save a Kaggle version
with outputs before session storage is lost. Failures stop subsequent runs;
no auto-resume/skip or overwrite. An estimated4-5h is not a runtime limit.


## 1. Latest source and dependencies


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['checkout', 'install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in []:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/kaggle/working/gpw_source')
os.environ["GIT_TERMINAL_PROMPT"] = "0"
if ROOT.exists():
    top = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "--show-toplevel"], text=True).strip()
    if Path(top).resolve() != ROOT.resolve():
        raise ValueError("Existing checkout must be its own Git repository")
    remote = subprocess.check_output(["git", "-C", str(ROOT), "remote", "get-url", "origin"], text=True).strip()
    branch = subprocess.check_output(["git", "-C", str(ROOT), "branch", "--show-current"], text=True).strip()
    if remote != REPOSITORY or branch != BRANCH:
        raise ValueError("Existing checkout repository or branch does not match the requested source")
    subprocess.run(["git", "-C", str(ROOT), "diff", "--exit-code", "HEAD"], check=True)
    untracked = subprocess.check_output(["git", "-C", str(ROOT), "ls-files", "--others",
        "--exclude-standard", "--", ":(glob)src/**/*.py", ":(glob)experiments/**/*.py"], text=True).strip()
    if untracked:
        raise ValueError(f"Untracked source files prevent verified reuse: {untracked}")
    subprocess.run(["git", "-C", str(ROOT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge-base", "--is-ancestor", "HEAD", "origin/" + BRANCH], check=True)
    subprocess.run(["git", "-C", str(ROOT), "merge", "--ff-only", "origin/" + BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH,
                    "--single-branch", REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)

COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
for path in reversed((str(ROOT / "src"), str(ROOT))):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_OISST_STAGES['checkout'] = True
_GPW_CELL_RESULT


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['install', 'settings', 'preflight', 'data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['checkout']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[data,ukhousing,benchmark]"], check=True)
import runpy
run_worker = runpy.run_path(str(ROOT / "experiments/colab_client.py"))["run_worker"]
WORK_ROOT = Path("/kaggle/working/gpw_results")
DATA_ROOT = Path("/kaggle/working/gpw_data")

def call_worker(action, *, on_event=None, **values):
    return run_worker(ROOT, {"action": action, "source_provenance": SOURCE_PROVENANCE, **values},
        module="experiments.multidataset_worker", jobs_directory=WORK_ROOT / "worker_jobs", on_event=on_event)
_OISST_STAGES['install'] = True


## 2. Dataset/model selection and all call-time overrides

Edit DATASETS to choose/order datasets. MODELS_BY_DATASET selects/orders
model names shown in AVAILABLE_MODELS; an absent entry means the complete
displayed preset minus EXCLUDED_MODEL_TYPES. VNNGP is excluded by default;
an explicit selection conflicting with an exclusion raises.
Unknown names raise.
MODEL_OVERRIDES is {dataset: {model_name: {field: value}}}.
DATA_OVERRIDES is {dataset: {field: value}}; no source/schema relabeling.

Example short selection: DATASETS=["california"] and
MODELS_BY_DATASET={"california": ["SVGP", "GPW"]}.
For a long run, explicitly change EPOCHS and WARMUP_EPOCHS; no automatic
warm-up shrinking occurs. Rerunning training always starts fresh.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['settings', 'preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
from datetime import UTC, datetime
from IPython.display import FileLink, Image, Markdown, display
import pandas as pd

INPUT_ROOT = "/kaggle/input/datasets/chahinenejma/gpw-kaggle-inputs"
DATASETS = ['california', 'protein', 'tco', 'oisst']
EXCLUDED_MODEL_TYPES = ["vnngp"]
MODELS_BY_DATASET = {
    "california": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "protein": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "tco": [
        "SVGP",
        "GPW",
        "GPW (alternative)",
        "ProSpar",
        "ProSpar (alternative)",
        "Standard Patchwork (PC tree)"
    ],
    "oisst": [
        "SVGP",
        "GPW",
        "ProSpar",
        "Standard Patchwork (PC tree)"
    ]
}
MODEL_OVERRIDES = {
    "california": {
        "ProSpar": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        },
        "ProSpar (alternative)": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        }
    },
    "protein": {
        "ProSpar": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        },
        "ProSpar (alternative)": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        }
    },
    "tco": {
        "ProSpar": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        },
        "ProSpar (alternative)": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        }
    },
    "oisst": {
        "ProSpar": {
            "prospar": {
                "temperatures": 1.0,
                "exponent": 1.0,
                "inducing_nuggets": 0.01,
                "prior_jitter": 0.0
            }
        }
    }
}
DATA_OVERRIDES = {}
EPOCHS = 100
WARMUP_EPOCHS = 20
EARLY_STOPPING = {'patience': 10, 'relative_min_improvement': 0.001}
SAVE_INDUCING_POSITIONS = True
SEEDS = [32, 33]
PROGRESS_EVERY_EPOCHS = 20

settings = call_worker("settings", study="benchmark_repeats", seeds=SEEDS, progress_every_epochs=PROGRESS_EVERY_EPOCHS,
    device="cuda:0", input_root=INPUT_ROOT, datasets=DATASETS,
    excluded_model_types=EXCLUDED_MODEL_TYPES,
    early_stopping=EARLY_STOPPING,
    models=MODELS_BY_DATASET, model_overrides=MODEL_OVERRIDES, data_overrides=DATA_OVERRIDES,
    epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS, save_inducing_positions=SAVE_INDUCING_POSITIONS)
PLAN, PLAN_SHA256 = settings["plan"], settings["plan_sha256"]
AVAILABLE_MODELS = settings["available"]
display(pd.DataFrame(settings["summary"]))
print("Total selected model runs:", settings["total_runs"])
print("Total training epochs across runs:", settings["total_epochs"])
print("Deferred datasets:", PLAN["deferred"])
display(pd.DataFrame([{"dataset": row["dataset"], "model": row["name"], **row["config"]} for row in PLAN["runs"]]))
_GPW_CELL_RESULT = print("Available model names:", AVAILABLE_MODELS)
_OISST_STAGES['settings'] = True
_GPW_CELL_RESULT


## 3. Check attached inputs, requested GPU and available disk


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['preflight', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREFLIGHT_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("preflight_%Y%m%dT%H%M%S%fZ")
preflight = call_worker("preflight", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREFLIGHT_DIRECTORY))
_GPW_CELL_RESULT = print(preflight)
_OISST_STAGES['preflight'] = True
_GPW_CELL_RESULT


## 4. Prepare selected datasets once or explicitly load prepared files

Leave this stage untouched when only model settings change. Use
PREPARED_ACTION="load" and an exact PREPARED_PATH to reuse saved arrays;
configuration/checksum mismatches raise without rebuilding or filling.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['data', 'training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
PREPARED_ACTION = "prepare"
PREPARED_PATH = None
if PREPARED_ACTION == "prepare":
    PREPARED_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("prepared_%Y%m%dT%H%M%S%fZ")
    prepared = call_worker("prepare", plan=PLAN, plan_sha256=PLAN_SHA256,
        directory=str(PREPARED_DIRECTORY), data_root=str(DATA_ROOT))
elif PREPARED_ACTION == "load":
    if PREPARED_PATH is None:
        raise ValueError("Set PREPARED_PATH explicitly for verified prepared-data reuse")
    PREPARED_DIRECTORY = Path(PREPARED_PATH)
    prepared = call_worker("load", plan=PLAN, plan_sha256=PLAN_SHA256, directory=str(PREPARED_DIRECTORY))
else:
    raise ValueError("PREPARED_ACTION must be prepare or load")
counts = []
for spec in PLAN["data"]:
    import numpy as np
    with np.load(PREPARED_DIRECTORY / spec["name"] / "split.npz", allow_pickle=False) as rows:
        counts.append({"dataset": spec["name"], **{label: len(rows[label]) for label in ("train", "validation", "test")}})
display(pd.DataFrame(counts))
_GPW_CELL_RESULT = print("Prepared datasets:", prepared["directory"])
_OISST_STAGES['data'] = True
_GPW_CELL_RESULT


## 5. Explicitly authorize and run the displayed plan

RUN_TRAINING=False intentionally raises before any model starts. Set it
True only after reviewing selection, counts and capacities above. There
is no endless sweep: exactly the selected configurations and epoch counts
run. Any failure stops subsequent work with a full traceback. Interrupting
the notebook terminates the worker. Completed results/ZIPs remain on disk.
Changing the selection/hyperparameters requires rerunning settings and
preflight; unchanged prepared data can be reused explicitly.


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['training', 'archive']:
    _OISST_STAGES[_stage] = False
for _required in ['install', 'settings', 'preflight', 'data']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
RUN_TRAINING = False
if RUN_TRAINING is not True:
    raise RuntimeError("Training disabled. Review the plan, then explicitly set RUN_TRAINING=True")
live = display("Starting selected experiments...", display_id=True)

def show_event(event):
    path = Path(event["directory"])
    if event["kind"] == "snapshot":
        live.update(Image(filename=str(path / "visuals/training.png")))
    elif event["kind"] == "complete":
        display(Markdown("### " + event["dataset"] + " / " + event["name"]))
        display(pd.DataFrame([event["metrics"]]).T)
        display(Image(filename=str(path / "visuals/predictions.png")))
        display(Image(filename=str(path / "visuals/hyperparameters.png")))
        metadata = json.loads((path / "model_metadata.json").read_text())
        if metadata["inducing_positions_saved"]:
            display(Image(filename=str(path / "visuals/inducing_positions.png")))
        print("Saved completed-run archive:", event["archive"])
        display(FileLink(event["archive"]))
    else:
        raise ValueError(f"Unknown worker event: {event['kind']}")

BATCH_DIRECTORY = WORK_ROOT / datetime.now(UTC).strftime("experiments_%Y%m%dT%H%M%S%fZ")
result = call_worker("train", plan=PLAN, plan_sha256=PLAN_SHA256,
    prepared_directory=str(PREPARED_DIRECTORY), directory=str(BATCH_DIRECTORY),
    execute_training=RUN_TRAINING, on_event=show_event)
display(pd.read_csv(Path(result["directory"]) / "comparison.csv"))
_GPW_CELL_RESULT = print("Complete collection:", result["directory"])
_OISST_STAGES['training'] = True
_GPW_CELL_RESULT


## 6. Archive the completed collection


In [ ]:
if "_OISST_STAGES" not in globals():
    _OISST_STAGES = {}
for _stage in ['archive']:
    _OISST_STAGES[_stage] = False
for _required in ['training']:
    if not _OISST_STAGES.get(_required, False):
        raise RuntimeError(f"Run the {_required} stage successfully first")
ARCHIVE_PATH = WORK_ROOT / datetime.now(UTC).strftime("all_results_%Y%m%dT%H%M%S%fZ.zip")
archived = call_worker("archive", plan=PLAN, plan_sha256=PLAN_SHA256,
    directory=str(BATCH_DIRECTORY), destination=str(ARCHIVE_PATH))
from IPython.display import FileLink
display(FileLink(archived["archive"]))
print("Worker requests/logs/failures:", WORK_ROOT / "worker_jobs")
_GPW_CELL_RESULT = print("Save a Kaggle version with outputs to preserve all files.")
_OISST_STAGES['archive'] = True
_GPW_CELL_RESULT
